# 04 — RQ4: Vector DB Choice (Qdrant vs ChromaDB)

**Câu hỏi:** Tại sao chọn Qdrant? Nó nhanh và mạnh hơn các giải pháp khác ở điểm nào?

**Metric:** Latency (Search), Latency (Filtered Search).

In [1]:
# --- Setup ---
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json

HERE = Path.cwd()
for p in [HERE] + list(HERE.parents):
    if (p / "source").is_dir() and (p / "research").is_dir():
        TRAFFIC_RAG = p
        break

RESULTS_DIR = TRAFFIC_RAG / "research" / "results" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import qdrant_client
import chromadb
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

CHUNKS_PATH = TRAFFIC_RAG / "Data" / "all_chunks.jsonl"
chunks = [json.loads(l) for l in CHUNKS_PATH.open(encoding='utf-8') if l.strip()][:500]


## 1. Prepare Data & Benchmarking

In [2]:
# Initialize clients
q_client = QdrantClient(host="localhost", port=6334)
c_client = chromadb.EphemeralClient()
collection_name = "benchmark_test"

# Mock embedding (384 dimensions for sbert)
def mock_embedding(): return np.random.rand(384).tolist()

# --- Qdrant Setup ---
q_client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)
points = [
    PointStruct(
        id=i, vector=mock_embedding(), 
        payload={"text": c['content'], "doc_id": c['metadata']['doc_id'], "dieu": c['metadata'].get('dieu', 0)}
    )
    for i, c in enumerate(chunks)
]
q_client.upsert(collection_name=collection_name, points=points)

# --- ChromaDB Setup ---
chroma_coll = c_client.get_or_create_collection(name=collection_name)
chroma_coll.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=[mock_embedding() for _ in chunks],
    metadatas=[{"doc_id": c['metadata']['doc_id'], "dieu": c['metadata'].get('dieu', 0)} for c in chunks],
    documents=[c['content'] for c in chunks]
)


/tmp/ipykernel_228453/447527318.py:2: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  q_client = QdrantClient(host="localhost", port=6334)
/tmp/ipykernel_228453/447527318.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  q_client.recreate_collection(


In [3]:
def benchmark_search(db_type):
    vec = mock_embedding()
    t0 = time.perf_counter()
    for _ in range(50):
        if db_type == 'qdrant':
            q_client.search(collection_name=collection_name, query_vector=vec, limit=10)
        else:
            chroma_coll.query(query_embeddings=[vec], n_results=10)
    return (time.perf_counter() - t0) / 50

def benchmark_filtered(db_type):
    vec = mock_embedding()
    t0 = time.perf_counter()
    for _ in range(50):
        if db_type == 'qdrant':
            from qdrant_client.models import Filter, FieldCondition, MatchValue
            q_client.search(
                collection_name=collection_name, 
                query_vector=vec, 
                query_filter=Filter(must=[FieldCondition(key="dieu", match=MatchValue(value=6))]),
                limit=10
            )
        else:
            chroma_coll.query(query_embeddings=[vec], n_results=10, where={"dieu": 6})
    return (time.perf_counter() - t0) / 50

results = []
for db in ['qdrant', 'chroma']:
    results.append({
        'db': db,
        'search_latency': benchmark_search(db),
        'filtered_latency': benchmark_filtered(db)
    })

df = pd.DataFrame(results)
print(df)
df.to_csv(RESULTS_DIR / 'rq4_vectordb.csv', index=False)


/tmp/ipykernel_228453/1502169753.py:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  q_client.search(collection_name=collection_name, query_vector=vec, limit=10)


/tmp/ipykernel_228453/1502169753.py:17: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  q_client.search(


       db  search_latency  filtered_latency
0  qdrant        0.015402          0.003721
1  chroma        0.001988          0.001190
